## Generating th elow level datasets based on capture json files

In [10]:
import json
import pandas as pd
import numpy as np
import os
from pathlib import Path

def list_files_recursively(directory, extension=".json"):
    directory = Path(directory)
    files = []
    for path in directory.rglob(f"*{extension}"):
        files.append(path)
    return files



In [11]:
# Define input and output directories
git_directory_path = r'C:\Users\vassa\Desktop\UZH\Masters Project\synthetic_network_data_gen\captures\captures_json'
output_dir = r"C:\Users\vassa\Desktop\UZH\Masters Project\synthetic_network_data_gen\low_level_features\dataset\separate files"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Get all JSON files in the input directory
json_files = list_files_recursively(git_directory_path, extension=".json")
print(f"Found {len(json_files)} JSON files in {git_directory_path}")



Found 11918 JSON files in C:\Users\vassa\Desktop\UZH\Masters Project\synthetic_network_data_gen\captures\captures_json


In [12]:
def as_list_on_duplicate_keys(ordered_pairs):
    """
    A custom JSON object_pairs_hook that collects values for duplicate
    keys into a list.
    """
    d = {}
    for k, v in ordered_pairs:
        if k in d:
            if isinstance(d[k], list):
                d[k].append(v)
            else:
                d[k] = [d[k], v]
        else:
            d[k] = v
    return d

In [13]:
def load_json_file(file_path):
    """Load a single JSON file with duplicate key handling"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            pcap_data = json.load(f, object_pairs_hook=as_list_on_duplicate_keys)
        print(f"Successfully loaded {len(pcap_data)} packets from {file_path}")
        return pcap_data
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return []
    except Exception as e:
        try:
            with open(file_path, 'r', encoding='utf-16') as f:
                pcap_data = json.load(f, object_pairs_hook=as_list_on_duplicate_keys)
            print(f"Successfully loaded {len(pcap_data)} packets from {file_path}")
            return pcap_data
        except Exception as e:  
            print(f"Error loading {file_path}: {e}")
            return []

In [14]:
# === Helper function to safely get nested fields ===
def get(d, path, default=None):
    """Safely get nested dictionary values using dot notation."""
    for p in path.split('.'):
        if isinstance(d, dict) and p in d:
            d = d[p]
        else:
            return default
    return d

In [15]:
def extract_packet_features(packets):
    """
    Extracts low-level features from a list of packet data.
    Handles multiple QUIC sections properly.
    Uses counters instead of binary flags for frame types.
    """
    all_packets_features = []

    if not packets:
        return []

    # Determine server IP from the first two packets to establish direction
    # Assumes the server is the destination of the first packet sent by the client.
    try:
        server_ip = packets[0]['_source']['layers']['ip']['ip.dst']
        print(f"Server IP identified as: {server_ip}")
    except (KeyError, IndexError):
        print("Could not determine server IP automatically. Please check the JSON data.")
        return []

    for packet_data in packets:
        try:
            layers = packet_data['_source']['layers']
            frame_info = layers.get('frame', {})
            ip_info = layers.get('ip', {})
            
            # Handle QUIC layers - could be single dict or list of dicts
            quic_raw = layers.get('quic')
            if quic_raw is None:
                # No QUIC layer, skip this packet
                continue
                
            # Normalize to list format
            if isinstance(quic_raw, list):
                quic_sections = quic_raw
            else:
                quic_sections = [quic_raw]
            
            # Collect frames from all sections
            all_frames = []
            for quic_section in quic_sections:
                frames = quic_section.get('quic.frame', [])
                if not isinstance(frames, list):
                    frames = [frames] if frames else []
                all_frames.extend(frames)
            
            # --- Basic Features ---
            delta_time = float(frame_info.get('frame.time_delta', 0.0))
            packet_length = int(frame_info.get('frame.len', 0))
            frame_number = int(frame_info.get('frame.number', 0))
            packet_direction = 1 if ip_info.get('ip.src') == server_ip else 0

            # --- QUIC Header Features ---
            # Check if ANY section has a long header
            has_any_long_header = any('quic.long.packet_type' in section for section in quic_sections)
            header_form = 1 if has_any_long_header else 0
            
            # --- QUIC Packet Type Counters ---
            # Count ALL sections for packet types - a packet can contain multiple types
            count_initial, count_0rtt, count_handshake, count_retry, count_vn, count_1rtt = 0, 0, 0, 0, 0, 0
            
            # Check each QUIC section for packet types
            for quic_section in quic_sections:
                # Check for version negotiation
                if quic_section.get('quic.version') == '0x00000000':
                    count_vn += 1
                
                # Check for long header packet types
                if 'quic.long.packet_type' in quic_section:
                    pkt_type = quic_section.get('quic.long.packet_type')
                    if pkt_type == '0': 
                        count_initial += 1
                    elif pkt_type == '1': 
                        count_0rtt += 1
                    elif pkt_type == '2': 
                        count_handshake += 1
                    elif pkt_type == '3': 
                        count_retry += 1
                else:
                    # Short header = 1-RTT packet (check if this section has short header indicators)
                    if 'quic.short' in quic_section or not has_any_long_header:
                        count_1rtt += 1

            # --- QUIC Frame-based Features (Counters) ---
            count_path_challenge, count_path_response, count_new_cid, count_retire_cid = 0, 0, 0, 0
            count_padding, count_ack, count_close = 0, 0, 0
            count_ping = 0
            count_crypto, count_handshake_done = 0, 0
            
            http3_stream_ids = set()
            http3_fin_count = 0
            stream_length = 0
            stream_types = set()

            # Process all frames from all QUIC sections
            for frame in all_frames:
                try:
                    frame_type_str = frame.get('quic.frame_type', '0x-1')
                    if frame_type_str.startswith('0x'):
                        frame_type = int(frame_type_str, 16)
                    else:
                        frame_type = int(frame_type_str)

                    if frame_type == 0x00: 
                        count_padding += 1
                    if frame_type == 0x01: 
                        count_ping += 1
                    elif frame_type in [0x02, 0x03]: 
                        count_ack += 1
                    elif frame_type in [0x1c, 0x1d]: 
                        count_close += 1
                    elif frame_type == 0x18: 
                        count_new_cid += 1
                    elif frame_type == 0x19: 
                        count_retire_cid += 1
                    elif frame_type == 0x1a: 
                        count_path_challenge += 1
                    elif frame_type == 0x1b: 
                        count_path_response += 1
                    elif frame_type == 0x06:  # CRYPTO frame
                        count_crypto += 1
                    elif frame_type == 0x1e:  # HANDSHAKE_DONE frame
                        count_handshake_done += 1
                    
                    # Stream related features
                    if 0x08 <= frame_type <= 0x0f:
                        stream_id = frame.get('quic.stream.stream_id')
                        if stream_id:
                            http3_stream_ids.add(stream_id)
                            
                            # Add stream type based on stream ID
                            try:
                                sid = int(stream_id)
                                # QUIC stream types: 0=client-initiated bidirectional, 1=server-initiated bidirectional
                                # 2=client-initiated unidirectional, 3=server-initiated unidirectional
                                stream_type = sid & 0x03  # Get last 2 bits
                                stream_types.add(stream_type)
                            except (ValueError, TypeError):
                                pass
                        
                        # Check for FIN bit
                        fin_bit = frame.get('quic.frame_type_tree', {}).get('quic.stream.fin', '0')
                        if fin_bit == '1':
                            http3_fin_count += 1
                            
                        # Calculate stream data length
                        try:
                            data_len = frame.get('quic.stream.length')
                            if data_len:
                                stream_length += int(data_len)
                            
                            else:
                                stream_data = frame.get('quic.stream_data')
                                if stream_data:
                                    stream_length += len(stream_data.replace(':', '')) // 2  # Hex string to bytes
                        except (ValueError, TypeError):
                            print(ValueError, TypeError)
                            pass
                            
                except (ValueError, TypeError) as e:
                    # Skip frames with invalid frame types
                    continue
            
            http3_stream_count = len(http3_stream_ids)
            stream_type_count = len(stream_types)  # Number of different stream types in this packet
            
            # Append all extracted features for this packet
            features = {
                'delta_time': delta_time,
                'packet_length': packet_length,
                'frame_number': frame_number,
                'packet_direction': packet_direction,
                'header_form': header_form,
                'count_initial': count_initial,
                'count_0rtt': count_0rtt,
                'count_handshake': count_handshake,
                'count_1rtt': count_1rtt,
                'count_retry': count_retry,
                'count_vn': count_vn,
                'count_path_challenge': count_path_challenge,
                'count_path_response': count_path_response,
                'count_new_connection_id': count_new_cid,
                'count_retire_cid': count_retire_cid,
                'count_ping': count_ping,
                'count_padding': count_padding,
                'count_ack': count_ack,
                'count_connection_close': count_close,
                'count_crypto': count_crypto,
                'count_handshake_done': count_handshake_done,
                'http3_stream_count': http3_stream_count,
                'http3_fin_count': http3_fin_count,
                'stream_length': stream_length,
                'stream_type_count': stream_type_count
            }
            all_packets_features.append(features)

        except Exception as e:
            packet_num = frame_info.get('frame.number', 'N/A')
            print(f"Could not process packet {packet_num}. Error: {e}. Skipping...")

    return all_packets_features

In [16]:
# Process all JSON files and create corresponding CSV files
successful_files = 0
failed_files = 0

for json_file in json_files:
    try:
        print(f"\n--- Processing {json_file.name} ---")
        
        # Load the JSON data
        pcap_data = load_json_file(json_file)
        
        if not pcap_data:
            print(f"Skipping {json_file.name} - no data loaded")
            failed_files += 1
            continue
        
        # Extract features
        extracted_features = extract_packet_features(pcap_data)
        
        if extracted_features:
            # Define the desired order of columns for the CSV file
            column_order = [
                'frame_number', 'delta_time', 'packet_length', 'packet_direction', 'header_form',
                'count_initial', 'count_0rtt', 'count_handshake', 'count_1rtt', 'count_retry', 'count_vn',
                'count_ack', 'count_padding', 'count_connection_close',
                'count_path_challenge', 'count_path_response', 
                'count_new_connection_id', 'count_retire_cid', 'count_ping',
                'count_crypto', 'count_handshake_done',
                'http3_stream_count', 'http3_fin_count', 'stream_length', 'stream_type_count'
            ]

            # Create a pandas DataFrame from the list of feature dictionaries
            features_df = pd.DataFrame(extracted_features)
            
            # Only reorder columns that exist in the DataFrame
            available_columns = [col for col in column_order if col in features_df.columns]
            features_df = features_df[available_columns]

            # Create output filename (replace .json with .csv)
            csv_filename = json_file.stem + '.csv'
            csv_output_path = os.path.join(output_dir, csv_filename)
            
            # Save the DataFrame to a CSV file
            features_df.to_csv(csv_output_path, index=False)

            print(f"✓ Extracted {len(extracted_features)} packets and saved to '{csv_output_path}'")
            successful_files += 1
            
        else:
            print(f"✗ No features were extracted from {json_file.name}")
            failed_files += 1
            
    except Exception as e:
        print(f"✗ Error processing {json_file.name}: {e}")
        failed_files += 1

print(f"\n=== Processing Complete ===")
print(f"Successfully processed: {successful_files} files")
print(f"Failed to process: {failed_files} files")
print(f"Output directory: {output_dir}")

# Display summary of one file as example (if any successful)
if successful_files > 0:
    # Load the first successfully created CSV for display
    first_csv = list(Path(output_dir).glob('*.csv'))[0]
    print(f"\nExample output from {first_csv.name}:")
    example_df = pd.read_csv(first_csv)
    display(example_df)


--- Processing 3172_aioquic_before_fast.json ---
Successfully loaded 14 packets from C:\Users\vassa\Desktop\UZH\Masters Project\synthetic_network_data_gen\captures\captures_json\aioquic\aioquic\3172_aioquic_before_fast.json
Server IP identified as: 127.0.0.1
✓ Extracted 14 packets and saved to 'C:\Users\vassa\Desktop\UZH\Masters Project\synthetic_network_data_gen\low_level_features\dataset\separate files\3172_aioquic_before_fast.csv'

--- Processing 3173_aioquic_before_fast.json ---
Successfully loaded 14 packets from C:\Users\vassa\Desktop\UZH\Masters Project\synthetic_network_data_gen\captures\captures_json\aioquic\aioquic\3173_aioquic_before_fast.json
Server IP identified as: 127.0.0.1
✓ Extracted 14 packets and saved to 'C:\Users\vassa\Desktop\UZH\Masters Project\synthetic_network_data_gen\low_level_features\dataset\separate files\3173_aioquic_before_fast.csv'

--- Processing 3174_aioquic_before_fast.json ---
Successfully loaded 14 packets from C:\Users\vassa\Desktop\UZH\Masters P

,frame_number,delta_time,packet_length,packet_direction,header_form,count_initial,count_0rtt,count_handshake,count_1rtt,count_retry,...,count_path_response,count_new_connection_id,count_retire_cid,count_ping,count_crypto,count_handshake_done,http3_stream_count,http3_fin_count,stream_length,stream_type_count
0,1,0.000000,1242,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,2,0.000087,89,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,3,0.000554,1242,0,1,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,4,0.001087,1242,1,1,1,0,1,0,0,...,0,0,0,0,4,0,0,0,0,0
4,5,0.000021,403,1,1,0,0,1,0,0,...,0,0,0,0,2,0,0,0,0,0
5,6,0.003077,1392,0,1,1,0,1,1,0,...,0,1,0,0,1,0,0,0,0,0
6,7,0.000251,623,1,0,0,0,0,1,0,...,0,1,0,0,2,1,2,0,10,1
7,8,0.000153,85,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
8,9,0.000098,1392,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
9,10,0.000032,1242,1,0,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,0


In [16]:
pd.options.display.max_columns = None